In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\Jack\Downloads\Updated_Balanced_Food_Items_Dataset.csv")
df.sample(10)

,Food Name,Category,Label,Exclude
626,Dumpling chive pork,Prepared Foods,Not Junk,No
1088,"Diet Coke, Caffeine Free",Beverages,Junk,No
553,Prosciutto,Pork Products,Not Junk,No
139,"Diet Coke, Caffeine Free",Beverages,Junk,No
595,"Potato, Boiled with Skin",Vegetables and Vegetable Products,Not Junk,No
97,"Carrots, Cooked From Fresh",Vegetables and Vegetable Products,Not Junk,No
1163,"Peanuts, Raw",Nut and Seed Products,Not Junk,No
1193,Butter Pecan Ice Cream,Snacks or Sweets,Junk,No
1156,"Broccoli, Cooked From Fresh",Vegetables and Vegetable Products,Not Junk,No
339,"Beef Steak, Sirloin, No Visible Fat Eaten",Beef Products,Not Junk,No


Convert labels to integers (0 for Not Junk, 1 for Junk)

In [3]:
df["Category"] = df["Category"].str.lower()
df["Food Name"] = df["Food Name"].str.lower()
# Combine 'Food Name' and 'Category' for tokenization
df['Combined'] = df['Food Name'] + ' - ' + df['Category']
label_map = {"Not Junk":0, "Junk":1}
df["Label"] = df["Label"].map(label_map)

BERT requires input in a specific format (tokenized text), so we’ll use the BertTokenizer to preprocess the food item descriptions.

In [4]:
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenize the data
def encode_data(texts, labels, tokenizer):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=64)
    return encodings, labels


c:\Python\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Now, create a custom PyTorch Dataset to return the tokenized data in the format BERT expects.

In [5]:
import torch
from torch.utils.data import Dataset

class FoodDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=64)
        self.labels = labels
    def __getitem__(self, index):
        item = {key: torch.tensor(val[index]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[index])
        return item
    def __len__(self):
        return len(self.labels)

# Prepare the dataset
training_texts = df["Food Name"].tolist()
training_label = df["Label"].tolist()

# Create the dataset
training_dataset = FoodDataset(training_texts, training_label, tokenizer)

Check if GPU is available

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


Now, load a pre-trained BERT model with a classification head.

In [7]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

Move the model to the GPU

In [8]:
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

Next, we define the training arguments, such as the number of epochs, batch size, etc.

In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=16,   # batch size for training
    per_device_eval_batch_size=16,    # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    fp16=True                        # # Mixed precision for better performance on GPUs
)

Now, set up the Trainer class with the model, training arguments, and dataset.

In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model, 
    args=training_args,
    train_dataset=training_dataset,
    eval_dataset=None  # You can add validation data here if available
)

c:\Python\Lib\site-packages\accelerate\accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Finally, train the model using the Trainer class.

In [11]:
# Train the model
trainer.train()

  0%|          | 0/231 [00:00<?, ?it/s]

c:\Python\Lib\site-packages\transformers\models\bert\modeling_bert.py:439: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


{'loss': 0.6861, 'grad_norm': 5.408079147338867, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.13}
{'loss': 0.6619, 'grad_norm': 7.965125560760498, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.26}
{'loss': 0.6395, 'grad_norm': 5.661340236663818, 'learning_rate': 2.9e-06, 'epoch': 0.39}
{'loss': 0.5629, 'grad_norm': 4.557079792022705, 'learning_rate': 3.9e-06, 'epoch': 0.52}
{'loss': 0.4912, 'grad_norm': 6.076627731323242, 'learning_rate': 4.9000000000000005e-06, 'epoch': 0.65}
{'loss': 0.403, 'grad_norm': 5.968149185180664, 'learning_rate': 5.9e-06, 'epoch': 0.78}
{'loss': 0.3516, 'grad_norm': 4.379149913787842, 'learning_rate': 6.900000000000001e-06, 'epoch': 0.91}
{'loss': 0.2903, 'grad_norm': 6.251811981201172, 'learning_rate': 7.9e-06, 'epoch': 1.04}
{'loss': 0.2092, 'grad_norm': 4.224586486816406, 'learning_rate': 8.9e-06, 'epoch': 1.17}
{'loss': 0.1498, 'grad_norm': 6.319345474243164, 'learning_rate': 9.900000000000002e-06, 'epoch': 1.3}
{'loss': 0.1034, 'grad_norm':

TrainOutput(global_step=231, training_loss=0.2058862905739706, metrics={'train_runtime': 13.5342, 'train_samples_per_second': 271.312, 'train_steps_per_second': 17.068, 'total_flos': 50948989204320.0, 'train_loss': 0.2058862905739706, 'epoch': 3.0})

After training, you can use the model to classify new food items.

In [12]:
# Function to classify new items
def classify_food_item(food_name, category):
    # Combine 'Food Name' and 'Category' in the same way as during training
    combined_text = food_name + ' - ' + category
    
    # Move tokenizer output tensors to the GPU (if available)
    encoding = tokenizer(combined_text, truncation=True, padding=True, max_length=64, return_tensors='pt').to(device)
    
    # Ensure that the model and data are on the same device
    outputs = model(**encoding)  # Model is already on GPU
    logits = outputs.logits
    
    predicted_class = torch.argmax(logits, dim=1).item()
    return 'Junk' if predicted_class == 1 else 'Not Junk'



In [20]:
# model.eval()
# Example usage
new_item = "cucumber"
category = "vegetable"
classification = classify_food_item(new_item, category)
print(f"The item '{new_item}' is classified as: {classification}")

The item 'cucumber' is classified as: Not Junk


In [14]:
import pandas as pd

In [15]:
df = pd.read_csv(r"F:\Sparta\Pre-assignment\Airflow Project\NLP\unlabelled_training_data.csv")
df.head()
#df = pd.read_csv(r"C:\Users\Jack\Downloads\servings.csv")

,Food Name,Category
0,"Fage, Total, Greek Strained Yoghurt, 5% Fat",Dairy and Egg Products
1,"Bulk, Pure Whey Isolate, Pistachio Ice Cream",Supplements
2,"Eggs, Cooked",Dairy and Egg Products
3,"Centrum Advance, Multivitamin",Supplements
4,"Seven Seas, Omega 3 Capsules, Max Strength",Supplements


In [16]:
df["generated_label"] = df.apply(
    lambda row: classify_food_item(str(row["Food Name"]) if pd.notnull(row["Food Name"]) else "", 
                                   str(row["Category"]) if pd.notnull(row["Category"]) else ""), 
    axis=1
)

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 671 entries, 0 to 670
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Food Name        671 non-null    object
 1   Category         629 non-null    object
 2   generated_label  671 non-null    object
dtypes: object(3)
memory usage: 15.9+ KB


In [18]:
df.sample(8)

,Food Name,Category,generated_label
361,"White Rice, Steamed",Cereal Grains and Pasta,Not Junk
158,Prosciutto,Pork Products,Not Junk
480,"Figs, Fresh",Fruits and Fruit Juices,Not Junk
641,"Coffee, Prepared From Grounds",Beverages,Not Junk
275,"Diet Coke, Caffeine Free",Beverages,Junk
362,"Chicken Breast, Skin Removed Before Cooking",Poultry Products,Not Junk
310,Kimchi,Vegetables and Vegetable Products,Not Junk
199,"Nectarine, Fresh",Fruits and Fruit Juices,Not Junk
